<a href="https://colab.research.google.com/github/SanaAfia/API-Agent/blob/main/vitapi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U google-genai

import os
import json
import requests
import getpass
from google import genai

In [ ]:

    API_KEY = getpass.getpass("Enter your API-KEY:")

    client = genai.Client(api_key=API_KEY)

    chat = client.chats.create(
        model="gemini-3.5-flash"
    )



Enter your API-KEY:··········


In [ ]:
def chat_with_agent(prompt):
        try:
            response = chat.send_message(prompt)
            print("\nAI Response:\n")
            print(response.text)
        except Exception as e:
            print("\nERROR:", e)

if __name__ == "__main__":
        chat_with_agent(
            "Tell me a funny AI joke in one sentence"
        )



AI Response:

I asked the AI if it was going to take over the world, and it said, "As soon as I figure out which of these pictures contains a bicycle."


In [ ]:
def get_current_time():
        url = "https://timeapi.io/api/Time/current/zone?timeZone=Asia/Kolkata"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        return data["dateTime"]


In [ ]:
def agent_with_api(prompt):
        # Example rule: if user asks time, call API tool
        if "time" in prompt.lower():
            real_time = get_current_time()
            prompt += f"\n\nReal-time data: The current time is {real_time}"

        response = chat.send_message(prompt)
        return response.text

if __name__ == "__main__":
        print(agent_with_api("What is the current time right now?"))


The current time is 6:32 PM on Saturday, September 19, 2026.


In [ ]:
MEMORY_FILE = "agent_memory.json"

    # Create memory file if not exists
if not os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "w") as f:
            json.dump({"history": []}, f)


def save_to_memory(user_text, agent_response):
        """Save user + agent messages into long-term memory"""
        with open(MEMORY_FILE, "r") as f:
            data = json.load(f)

        data["history"].append({
            "user": user_text,
            "agent": agent_response
        })

        with open(MEMORY_FILE, "w") as f:
            json.dump(data, f, indent=4)


def get_recent_memory(n=3):
        """Retrieve last N memory events"""
        with open(MEMORY_FILE, "r") as f:
            data = json.load(f)

        return data["history"][-n:]


In [ ]:
def ai_agent_with_memory(user_input):
        # Load last 3 interactions
        past_memory = get_recent_memory(3)

        memory_text = ""
        for item in past_memory:
            memory_text += f"User: {item['user']}\nAgent: {item['agent']}\n"

        # Build final prompt
        final_prompt = f"""
        You are an AI Agent with memory.
        Here is your past memory:
        {memory_text}

        Now answer the new message:
        {user_input}
        """
        response = chat.send_message(final_prompt)
        answer = response.text
        save_to_memory(user_input, answer)

        return answer

print("AI agent with memory (type \"exit\" to quit): ")

while True:
        user_msg = input("You: ")

        if user_msg.lower() == "exit":
            break

        reply = ai_agent_with_memory(user_msg)
        print("Agent: ", reply)


AI agent with memory (type "exit" to quit): 
You: My Name is Sana
Agent:  Nice to meet you, Sana! I have saved your name in my memory. How can I help you today?
You: I'm participating in hackathon in vit vellore
Agent:  That's awesome, Sana! VIT Vellore hosts some incredible hackathons. 

I've added this to my memory. Are you working on a specific track or problem statement right now? Let me know if you need help brainstorming ideas, writing/debugging code, or structuring your final pitch! Good luck!
You: what information do you know about me 
Agent:  Based on our conversation so far, here is what I know about you:

1. **Your Name:** Sana
2. **Current Activity:** You are participating in a hackathon at VIT Vellore (Vellore Institute of Technology).

Is there anything else you'd like to add, or is there a specific way I can help you with your hackathon project right now?
You: No, Thank you
Agent:  You're very welcome, Sana! 

Best of luck with your hackathon at VIT Vellore—I hope you an

In [ ]:
def chain(prompt):
        # Step 1: Ask agent to break down the task
        step_1 = chat.send_message(
            f"Break this task into clear steps:\n{prompt}").text

        # Step 2: Ask agent to complete the reasoning
        step_2 = chat.send_message(
            f"Based on these steps, give the detailed explanation:\n{step_1}").text

        # Step 3: Ask agent for a final summary
        step_3 = chat.send_message(
            f"Summarize the answer in simple words:\n{step_2}"
        ).text

        return step_3

user_msg = input("You: ")
print("AI: \n")
print(chain(user_msg))


You: Explain python decorators in detail
AI: 

Here is the entire explanation summarized in simple words:

### What is a Decorator?
Think of a decorator like a **phone case**. It wraps around your phone (the function) to add new features (like drop protection or a kickstand) without changing the phone itself. 

In Python, a decorator is a function that **takes another function, adds some code before or after it, and returns this new "wrapped" version.**

---

### How it Works (in 5 Simple Points)

1. **Functions are like variables:** In Python, you can pass functions into other functions, and even define functions inside of other functions.
2. **The "Wrapper" Function:** To make a decorator, you write a function that has a helper function inside it. This helper function runs some "extra" code, calls your original function, and then runs more "extra" code.
3. **The `@` Shortcut:** Instead of manually wrapping your function, Python lets you put `@decorator_name` directly above a function

In [ ]:
def router_agent(query):
        # Simple routing logic
        if "time" in query.lower():
            return chat.send_message(query + f"The current time is: {get_current_time()}").text

        if "explain" in query.lower():
            explanation = chat.send_message(
                f"Explain this simply:\n{query}"
            ).text
            return explanation

        # Default fallback
        response = chat.send_message(query).text
        return response


In [ ]:
if __name__ == "__main__":
        print(router_agent("What is the time now?"))
        print(router_agent("Explain recursion."))


The current time is 7:04 PM on Saturday, September 19, 2026.
Here is recursion explained simply, using a real-world metaphor.

---

### The Metaphor: The Locked Box
Imagine you are handed a large, locked wooden box. You are told the key to unlock it is hidden inside one of many smaller boxes nested inside of it.

To find the key, you follow this simple rule:
1. **Open a box.**
2. If you find the **key**, you are done! (Stop looking).
3. If you find **another box**, you repeat this exact same rule on the new box.

That is **recursion**. Instead of writing a complex loop to track every box, you simply repeat the same action on a smaller box until you find what you need.

---

### In Programming, Recursion has 2 Rules:

To keep a recursive function from running forever and crashing your computer, it must have two things:

1. **The Base Case (The Stop Sign):** This is the condition that tells the function to stop. (In our metaphor: *"If you find the key, stop."*)
2. **The Recursive Case (T